In [4]:
from itertools import combinations
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,\
        ha,SingleStateAnsatz,create_single_machine,\
        create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,hi,E_fcis,\
        compute_qgt,sampler_info
import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
from pyscf import gto, scf, fci
import time
import logging
# ========== 你原有全局参数（直接复用） ==========

bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='6-31G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能：{exc:.4f} eV")
# ===================== NetKet 哈密顿量和采样器 =====================
ha = nkx.operator.from_pyscf_molecule(mol)

# 单系统希尔伯特空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=4,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
Hatree_Fock = hi.all_states()[0] #Array([0, 0, 0, 1, 0, 0, 0, 1], dtype=int8)


/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: Prefer the new nk.driver.VMC_SR over VMC which supports minSR and SPRING.

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV
H₂ FCI 基准能量
E0 = -1.06677527 Ha  |  激发能：0.0000 eV
E1 = -0.94683628 Ha  |  激发能：3.2637 eV
E2 = -0.66643104 Ha  |  激发能：10.8939 eV
E3 = -0.52057334 Ha  |  激发能：14.8629 eV


In [7]:
hf = hi.all_states()[0]
hf

Array([0, 0, 0, 1, 0, 0, 0, 1], dtype=int8)

In [6]:
comb2 = list(combinations(hf, 2))
print(comb2)  # [(0, 1)]

[(Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(1, dtype=int8)), (Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(1, dtype=int8)), (Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(1, dtype=int8)), (Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(1, dtype=int8)), (Array(0, dtype=int8), Array(1, dtype=int8)), (Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(0, dtype=int8)), (Array(0, dtype=int8), Array(1, dtype=int8)), (Array(1, dtype=int8), Array(0, dtype=int8)), (Array(1, dtype=int8), Array(0, dtype=int8)), (Array(1, dtype=int8), Array(0, dtype=int8)), (Array(1, dtype=int8), Array(1, d

In [5]:
import numpy as np
from itertools import combinations

def get_ccsd_excitations_from_hf(hf_state):
    """
    从 HF occupation bitstring 枚举 CCSD singles 和 doubles.

    返回：
        singles: [((i, a),), ...]
        doubles: [((i, a), (j, b)), ...]

    其中 (i, a) 表示 i occupied -> a virtual
    """
    hf_state = np.asarray(hf_state)
    n_spin_orbitals = hf_state.size
    assert n_spin_orbitals % 2 == 0

    nmo = n_spin_orbitals // 2

    single_moves = []

    # alpha block
    alpha_sites = np.arange(0, nmo)
    alpha_occ = alpha_sites[hf_state[alpha_sites] == 1]
    alpha_vir = alpha_sites[hf_state[alpha_sites] == 0]

    for i in alpha_occ:
        for a in alpha_vir:
            single_moves.append((int(i), int(a)))

    # beta block
    beta_sites = np.arange(nmo, 2 * nmo)
    beta_occ = beta_sites[hf_state[beta_sites] == 1]
    beta_vir = beta_sites[hf_state[beta_sites] == 0]

    for i in beta_occ:
        for a in beta_vir:
            single_moves.append((int(i), int(a)))

    # singles
    singles = [((i, a),) for i, a in single_moves]

    # doubles
    doubles = []
    for move1, move2 in combinations(single_moves, 2):
        i, a = move1
        j, b = move2

        # 两个电子不能来自同一个 occupied，也不能去同一个 virtual
        if i != j and a != b:
            doubles.append((move1, move2))

    return singles, doubles


Hatree_Fock = hi.all_states()[0]

singles, doubles = get_ccsd_excitations_from_hf(Hatree_Fock)

print("HF =", Hatree_Fock)
print()
print("Singles:")
for s in singles:
    print(s)

print()
print("Doubles:")
for d in doubles:
    print(d)

print()
print("n_singles =", len(singles))
print("n_doubles =", len(doubles))
print("total CCSD determinants =", 1 + len(singles) + len(doubles))

HF = [0 0 0 1 0 0 0 1]

Singles:
((3, 0),)
((3, 1),)
((3, 2),)
((7, 4),)
((7, 5),)
((7, 6),)

Doubles:
((3, 0), (7, 4))
((3, 0), (7, 5))
((3, 0), (7, 6))
((3, 1), (7, 4))
((3, 1), (7, 5))
((3, 1), (7, 6))
((3, 2), (7, 4))
((3, 2), (7, 5))
((3, 2), (7, 6))

n_singles = 6
n_doubles = 9
total CCSD determinants = 16


In [6]:
import numpy as np
from itertools import combinations

def get_ccsd_excitations_and_sampler_edges_from_hf(hf_state):
    """
    返回：
    1. sampler_edges: 给 NetKet Graph 用，格式为 [(i, a), ...]
    2. singles:       物理 single excitation，格式为 [((i, a),), ...]
    3. doubles:       物理 double excitation，格式为 [((i,a), (j,b)), ...]
    """
    hf_state = np.asarray(hf_state)
    n_spin_orbitals = hf_state.size
    assert n_spin_orbitals % 2 == 0

    nmo = n_spin_orbitals // 2

    sampler_edges = []

    # alpha block: 0 ~ nmo-1
    alpha_sites = np.arange(0, nmo)
    alpha_occ = alpha_sites[hf_state[alpha_sites] == 1]
    alpha_vir = alpha_sites[hf_state[alpha_sites] == 0]

    for i in alpha_occ:
        for a in alpha_vir:
            sampler_edges.append((int(i), int(a)))

    # beta block: nmo ~ 2*nmo-1
    beta_sites = np.arange(nmo, 2 * nmo)
    beta_occ = beta_sites[hf_state[beta_sites] == 1]
    beta_vir = beta_sites[hf_state[beta_sites] == 0]

    for i in beta_occ:
        for a in beta_vir:
            sampler_edges.append((int(i), int(a)))

    # 物理意义上的 single excitation
    singles = [(edge,) for edge in sampler_edges]

    # 物理意义上的 double excitation
    doubles = []
    for move1, move2 in combinations(sampler_edges, 2):
        i, a = move1
        j, b = move2

        # 不能动同一个电子，也不能占到同一个虚轨道
        if i != j and a != b:
            doubles.append((move1, move2))

    return sampler_edges, singles, doubles

In [8]:
sampler_edges, singles, doubles = get_ccsd_excitations_and_sampler_edges_from_hf(
    Hatree_Fock
)

In [9]:
sampler_edges

[(3, 0), (3, 1), (3, 2), (7, 4), (7, 5), (7, 6)]